In [ ]:
import features
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("hotel_bookings.csv")

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["is_canceled"].value_counts(normalize=True))

In [ ]:
# Missing value analysis
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": df.isna().mean() * 100
})

missing = missing[missing["Missing Count"] > 0]
missing = missing.sort_values(
    by="Missing Percentage",
    ascending=False
)

print(missing)

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=missing.reset_index(),
    x="Missing Percentage",
    y="index"
)

plt.xlabel("Missing Values (%)")
plt.ylabel("Features")
plt.title("Missing Values by Feature")
plt.tight_layout()
plt.show()

In [ ]:
# Check duplicate and inconsistent records

print("Duplicate rows:", df.duplicated().sum())

# Bookings with zero guests
zero_guests = (
    df["adults"].fillna(0)
    + df["children"].fillna(0)
    + df["babies"].fillna(0)
) == 0

print("Zero guests:", zero_guests.sum())

# Bookings with zero nights
zero_nights = (
    df["stays_in_weekend_nights"]
    + df["stays_in_week_nights"]
) == 0

print("Zero nights:", zero_nights.sum())

# Check unusual room prices
print("Negative ADR:", (df["adr"] < 0).sum())
print("ADR above 1000:", (df["adr"] > 1000).sum())

print("\nADR statistics:")
print(df["adr"].describe())

In [ ]:

# Step 3: Investigate suspicious records

# 1. Zero guests
zero_guests = (
    df["adults"].fillna(0)
    + df["children"].fillna(0)
    + df["babies"].fillna(0)
) == 0

# 2. Zero nights
zero_nights = (
    df["stays_in_weekend_nights"]
    + df["stays_in_week_nights"]
) == 0

print("=== Zero Guests ===")
print(pd.crosstab(
    df.loc[zero_guests, "hotel"],
    df.loc[zero_guests, "is_canceled"]
))

print("\n=== Zero Nights ===")
print(pd.crosstab(
    df.loc[zero_nights, "hotel"],
    df.loc[zero_nights, "is_canceled"]
))

print("\n=== Overlapping Issues ===")
print("Both zero guests and zero nights:",
      (zero_guests & zero_nights).sum())

# 3. Extreme room prices
print("\n=== Extreme ADR Records ===")
columns = [
    "hotel",
    "adr",
    "is_canceled",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "reservation_status"
]

print(
    df.loc[
        (df["adr"] < 0) | (df["adr"] > 1000),
        columns
    ].to_string(index=False)
)

In [ ]:
# Investigate the reservation status of zero-night bookings
pd.crosstab(
    df.loc[zero_nights, "reservation_status"],
    df.loc[zero_nights, "hotel"]
)

In [ ]:
# Work on a copy of the original dataset
df_features = df.copy()

# 1. Total nights
df_features["total_nights"] = (
    df_features["stays_in_weekend_nights"]
    + df_features["stays_in_week_nights"]
)

# 2. Total guests
df_features["total_guests"] = (
    df_features["adults"]
    + df_features["children"]
    + df_features["babies"]
)

# 3. Zero-night indicator
df_features["is_zero_night"] = (
    df_features["total_nights"] == 0
).astype(int)

# Inspect the engineered features
new_features = [
    "total_nights",
    "total_guests",
    "is_zero_night"
]

print(df_features[new_features].describe())
print("\nMissing values:")
print(df_features[new_features].isna().sum())

In [ ]:
# Investigate unusually large bookings

columns = [
    "hotel",
    "adults",
    "children",
    "babies",
    "total_guests",
    "total_nights",
    "is_canceled"
]

print("Largest bookings by guest count:")
print(
    df_features.nlargest(5, "total_guests")[columns]
)

print("\nLongest bookings:")
print(
    df_features.nlargest(5, "total_nights")[columns]
)

In [ ]:

# Selected booking-time features

features = [
    "hotel",
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "reserved_room_type",
    "deposit_type",
    "customer_type"
]

print("Selected features:", len(features))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Use the previously selected booking-time features
model_features = features + [
    "total_nights",
    "total_guests",
    "is_zero_night"
]

X = df_features[model_features].copy()
y = df_features["is_canceled"].copy()

# Keep identical feature vectors in the same split.
# Do not include the target or outcome-related columns.
groups = pd.util.hash_pandas_object(
    X, index=False
)

# Separate test set
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_val_idx, test_idx = next(
    split1.split(X, y, groups)
)

# Split remaining records into train and validation
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=42
)

relative_train, relative_val = next(
    split2.split(
        X.iloc[train_val_idx],
        y.iloc[train_val_idx],
        groups.iloc[train_val_idx]
    )
)

train_idx = train_val_idx[relative_train]
val_idx = train_val_idx[relative_val]

# Check actual sizes and class distributions
for name, idx in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx)
]:
    print(f"\n{name}: {len(idx)} rows")
    print(y.iloc[idx].value_counts(normalize=True))

# Ensure identical feature groups do not overlap
train_groups = set(groups.iloc[train_idx])
val_groups = set(groups.iloc[val_idx])
test_groups = set(groups.iloc[test_idx])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("\nGroup overlap check: PASSED")

In [ ]:

# Check the distribution of each split

splits = {
    "Train": train_idx,
    "Validation": val_idx,
    "Test": test_idx
}

for name, idx in splits.items():
    subset = df_features.iloc[idx]

    print(f"\n=== {name} ===")
    print("Records:", len(subset))
    print("Cancellation rate:",
          round(subset["is_canceled"].mean() * 100, 2), "%")

    print("\nHotel distribution:")
    print(subset["hotel"].value_counts(normalize=True) * 100)

In [ ]:
import numpy as np

np.savez(
    "split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

print("Split indices saved successfully.")

In [ ]:
# Create fixed train / validation / test datasets

train_df = df_features.iloc[train_idx].copy()
val_df = df_features.iloc[val_idx].copy()
test_df = df_features.iloc[test_idx].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
# Determine children imputation value using TRAINING DATA ONLY

children_fill_value = train_df["children"].mode()[0]

print("Children fill value:", children_fill_value)

In [ ]:
# Apply the training-derived value to all splits

for dataset in [train_df, val_df, test_df]:
    dataset["children"] = dataset["children"].fillna(children_fill_value)

    # Recalculate engineered total_guests
    dataset["total_guests"] = (
        dataset["adults"]
        + dataset["children"]
        + dataset["babies"]
    )

In [ ]:
print("Train missing children:",
      train_df["children"].isna().sum())

print("Validation missing children:",
      val_df["children"].isna().sum())

print("Test missing children:",
      test_df["children"].isna().sum())

print("Total guests missing:",
      train_df["total_guests"].isna().sum(),
      val_df["total_guests"].isna().sum(),
      test_df["total_guests"].isna().sum())

In [ ]:
final_features = [
    # Hotel type
    "hotel",

    # Booking timing
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",

    # Stay information
    "stays_in_weekend_nights",
    "stays_in_week_nights",

    # Guests
    "adults",
    "children",
    "babies",

    # Booking characteristics
    "meal",
    "country",
    "market_segment",
    "distribution_channel",

    # Customer history
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",

    # Booking type
    "reserved_room_type",
    "deposit_type",
    "customer_type",

    # Engineered features
    "total_nights",
    "total_guests",
    "is_zero_night"
]

target = "is_canceled"

X_train = train_df[final_features].copy()
X_val = val_df[final_features].copy()
X_test = test_df[final_features].copy()

y_train = train_df[target].copy()
y_val = val_df[target].copy()
y_test = test_df[target].copy()

print("Number of final features:", len(final_features))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [ ]:
categorical_features = [
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type"
]

numerical_features = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "total_nights",
    "total_guests",
    "is_zero_night"
]

print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))
print("Total:", len(categorical_features) + len(numerical_features))

In [ ]:
print("=== TRAIN ===")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\n=== VALIDATION ===")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("\n=== TEST ===")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Create separate copies
train_clean = df_features.iloc[train_idx].copy()
val_clean = df_features.iloc[val_idx].copy()
test_clean = df_features.iloc[test_idx].copy()

# 2. Calculate the missing-value replacement
# using training data only
children_fill = train_clean["children"].mode().iloc[0]

# 3. Apply the same cleaning rules to all sets
for dataset in [train_clean, val_clean, test_clean]:
    dataset["children"] = dataset["children"].fillna(
        children_fill
    )
    dataset["country"] = dataset["country"].fillna(
        "Unknown"
    )

    # Recalculate engineered features
    dataset["total_nights"] = (
        dataset["stays_in_weekend_nights"]
        + dataset["stays_in_week_nights"]
    )

    dataset["total_guests"] = (
        dataset["adults"]
        + dataset["children"]
        + dataset["babies"]
    )

    dataset["is_zero_night"] = (
        dataset["total_nights"] == 0
    ).astype(int)

# 4. Shared feature list
final_features = features + [
    "total_nights",
    "total_guests",
    "is_zero_night"
]

columns = final_features + ["is_canceled"]

train_clean = train_clean[columns]
val_clean = val_clean[columns]
test_clean = test_clean[columns]

# 5. Verify results
for name, dataset in [
    ("Train", train_clean),
    ("Validation", val_clean),
    ("Test", test_clean)
]:
    print(f"{name}: {dataset.shape}")
    print("Missing values:", dataset.isna().sum().sum())

In [ ]:
def feature_hashes(dataset):
    return set(pd.util.hash_pandas_object(
        dataset[final_features], index=False
    ))

train_hashes = feature_hashes(train_clean)
val_hashes = feature_hashes(val_clean)
test_hashes = feature_hashes(test_clean)

assert train_hashes.isdisjoint(val_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert val_hashes.isdisjoint(test_hashes)

print("Cleaned feature overlap check: PASSED")

In [ ]:
output_dir = Path("shared_data")
output_dir.mkdir(exist_ok=True)

train_clean.to_csv(
    output_dir / "train_clean.csv", index=False
)
val_clean.to_csv(
    output_dir / "val_clean.csv", index=False
)
test_clean.to_csv(
    output_dir / "test_clean.csv", index=False
)

np.savez(
    output_dir / "split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

print("Shared datasets saved successfully.")